# Cell report

Interactive results of `scripts/flow.sh <cell>`: schematic CACE run (PVT and Monte Carlo), DRC / antenna / LVS, extraction and the
schematic vs post-layout comparison (wire C only, and wire R + C). Generated by `scripts/build_report.sh`; the cell comes from the
environment variable `CELL` (default `ota_cmos5l`). Every plot is interactive: hover for values, drag to zoom, double-click to reset,
click a legend entry to hide it, use the drop-down to change the variable.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import init_notebook_mode
from IPython.display import HTML, Markdown, display

sys.path.insert(0, os.path.abspath("../scripts"))
import cace_data as cd

CELL = os.environ.get("CELL", "ota_cmos5l")
init_notebook_mode(connected=False)          # plotly.js embedded once -> the HTML works offline
pio.templates.default = "plotly_white"
CORNERS = ["tt", "ss", "ff", "sf", "fs"]
COLORS = dict(zip(CORNERS, ["#2a6fdb", "#d1495b", "#2e9e6b", "#e0a526", "#8a5cc2"]))
SYMBOLS = {1.2: "circle", 1.35: "square", 1.5: "diamond"}
st = cd.status(CELL)
ref = cd.find_run(CELL, "schematic")
df = cd.load_run(ref, CELL) if ref else pd.DataFrame()
lim = cd.limits(CELL)
display(HTML("<style>table.dataframe td, table.dataframe th {text-align:left !important; padding:2px 10px} table.dataframe{font-size:13px}</style>"))
display(Markdown(f"## {CELL}"))
display(Markdown(f"schematic run: `{os.path.relpath(ref, cd.ROOT) if ref else 'none'}`" + (f", flow run {st['generated']} (commit {st['commit']})" if st else "")))

## Status

In [ ]:
why = cd.stale(CELL)
if why:
    display(HTML("<div style='background:#fde8e8;border-left:4px solid #c0392b;padding:8px 12px;margin:6px 0'><b>Out of date:</b> " + "; ".join(why) + "</div>"))
if st:
    icon = {"pass": "✅", "fail": "❌", "error": "❗", "skip": "⏭", "blocked": "⛔"}
    t = pd.DataFrame(st["stages"])
    t["result"] = t["status"].map(icon).fillna("") + " " + t["status"]
    display(HTML(t[["name", "result", "seconds", "detail"]].to_html(index=False, escape=True)))
else:
    display(Markdown("No `docs/%s/status.json`: run `scripts/flow.sh %s`." % (CELL, CELL)))

## Schematic and layout
Circuit only (SPICE code blocks removed), light colour scheme; pictures are regenerated by `scripts/update_images.py`. The schematic is a vector drawing: use the browser zoom (Ctrl +) or open it in a new tab to read small text.

In [ ]:
from IPython.display import SVG, Image
im = cd.images(CELL)
info = cd.media_info(CELL)
if "schematic_svg" in im:
    display(Markdown("### Schematic"))
    display(SVG(filename=im["schematic_svg"]))
    rows = info.get("params") or []
    if rows:
        display(Markdown("**Schematic parameters** (`.param`, read from the schematic by `scripts/update_images.py`)"))
        pt = pd.DataFrame([dict(source=r["source"] + (" (" + ", ".join(r["instances"]) + ")" if r["instances"] else ""), parameter=r["name"], value=r["value"]) for r in rows])
        if pt.source.nunique() == 1 and not any(r["instances"] or r["source"] != CELL for r in rows):
            pt = pt[["parameter", "value"]]
        display(HTML(pt.to_html(index=False)))
else:
    display(Markdown("No schematic picture: run `scripts/update_images.py %s`." % CELL))
if "layout_png" in im:
    display(Markdown("### Layout"))
    display(Image(filename=im["layout_png"]))
    lay = info.get("layout")
    if lay:
        display(Markdown("**Physical size** (bounding box of `%s`)" % lay["top"]))
        display(HTML(pd.DataFrame([{"Width W [µm]": lay["W"], "Length L [µm]": lay["L"], "Area [µm²]": lay["area"]}]).to_html(index=False, float_format=lambda x: f"{x:.2f}")))
else:
    display(Markdown("No layout picture for this cell (no GDS in `cells.yaml`, or `scripts/update_images.py %s` was not run)." % CELL))

## Specification table (schematic, over PVT)

In [ ]:
if ref:
    s = cd.summary(ref)
    s["pass"] = s["status"].map({"Pass": "✅", "Fail": "❌", "Error": "❗"}).fillna("")
    show = s[["parameter", "variable", "min_limit", "min_value", "typ_value", "max_limit", "max_value", "pass"]]
    display(HTML(show.to_html(index=False)))

## PVT explorer
Each marker is one CACE run (corner = colour, supply = marker shape). Dashed lines are the specification limits of the datasheet.

In [ ]:
pvt = df[df.kind == "pvt"] if len(df) else df
if len(pvt):
    vars_ = list(dict.fromkeys(pvt.variable))
    fig = go.Figure()
    owner = []
    for v in vars_:
        d = pvt[pvt.variable == v]
        for c in CORNERS:
            for vdd, sym in SYMBOLS.items():
                x = d[(d.corner == c) & (d.vdd.round(2) == vdd)]
                if x.empty:
                    continue
                fig.add_trace(go.Scatter(x=x.temperature, y=x.value, mode="markers+lines", name=f"{c}", legendgroup=c,
                                         showlegend=(vdd == 1.2), visible=(v == vars_[0]), line=dict(width=1, color=COLORS[c]),
                                         marker=dict(symbol=sym, size=8, color=COLORS[c]),
                                         hovertemplate=f"{v}: %{{y:.5g}}<br>T=%{{x}} °C<br>{c}, {vdd} V<extra></extra>"))
                owner.append(v)
    def lines(v):
        lo, hi, unit, disp = lim.get(v, (None, None, "", v))
        sh = []
        for y in (lo, hi):
            if y is not None:
                sh.append(dict(type="line", xref="paper", x0=0, x1=1, y0=y, y1=y, line=dict(color="#c0392b", dash="dash", width=1.5)))
        return sh
    buttons = []
    for v in vars_:
        unit = lim.get(v, (None, None, "", v))[2] or pvt[pvt.variable == v].unit.iloc[0]
        disp = lim.get(v, (None, None, "", v))[3]
        buttons.append(dict(label=v, method="update",
                            args=[{"visible": [o == v for o in owner]},
                                  {"yaxis.title.text": f"{disp} [{unit}]", "shapes": lines(v), "title.text": f"{v} over PVT"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0.0, y=1.18, xanchor="left")],
                      xaxis_title="Temperature [°C]", yaxis_title=f"{lim.get(vars_[0], (0,0,'',vars_[0]))[3]} [{pvt[pvt.variable==vars_[0]].unit.iloc[0]}]",
                      title=dict(text=f"{vars_[0]} over PVT", x=0.5), shapes=lines(vars_[0]), height=520, margin=dict(t=90))
    fig.show()

### Worst cases over PVT

In [ ]:
if len(pvt):
    rows = []
    for v, d in pvt.groupby("variable", sort=False):
        lo, hi, unit, disp = lim.get(v, (None, None, "", v))
        a, b = d.loc[d.value.idxmin()], d.loc[d.value.idxmax()]
        mask = np.zeros(len(d), bool)
        if lo is not None:
            mask |= d.value.values < lo
        if hi is not None:
            mask |= d.value.values > hi
        bad = int(mask.sum())
        rows.append(dict(variable=v, unit=d.unit.iloc[0], min=a.value, min_at=f"{a.corner}, {a.vdd} V, {a.temperature:g} °C",
                         max=b.value, max_at=f"{b.corner}, {b.vdd} V, {b.temperature:g} °C", spec_min=lo, spec_max=hi, violations=f"{bad}/{len(d)}"))
    display(HTML(pd.DataFrame(rows).to_html(index=False, float_format=lambda x: f"{x:.5g}", na_rep="any")))

## Monte Carlo (mismatch)

In [ ]:
mc = df[df.kind == "mc"] if len(df) else df
if len(mc):
    vars_ = list(dict.fromkeys(mc.variable))
    fig = go.Figure(); owner = []
    for v in vars_:
        for c in CORNERS:
            x = mc[(mc.variable == v) & (mc.corner == c)]
            if x.empty:
                continue
            fig.add_trace(go.Histogram(x=x.value, name=c, legendgroup=c, opacity=0.55, marker_color=COLORS[c], visible=(v == vars_[0]),
                                       showlegend=True, nbinsx=30, hovertemplate=f"{c}: %{{x:.4g}} (%{{y}})<extra></extra>"))
            owner.append(v)
    unit = lambda v: mc[mc.variable == v].unit.iloc[0]
    fig.update_layout(barmode="overlay", height=480, margin=dict(t=90), title=dict(text=f"{vars_[0]} Monte Carlo", x=0.5), xaxis_title=f"{vars_[0]} [{unit(vars_[0])}]", yaxis_title="runs",
                      updatemenus=[dict(buttons=[dict(label=v, method="update", args=[{"visible": [o == v for o in owner]},
                                                  {"xaxis.title.text": f"{v} [{unit(v)}]", "title.text": f"{v} Monte Carlo"}]) for v in vars_],
                                        direction="down", x=0.0, y=1.18, xanchor="left")])
    fig.show()
    g = mc.groupby(["variable", "corner"]).value.agg(["count", "mean", "std", "min", "max"]).reset_index()
    g["mean-3σ"] = g["mean"] - 3 * g["std"]; g["mean+3σ"] = g["mean"] + 3 * g["std"]
    display(HTML(g.to_html(index=False, float_format=lambda x: f"{x:.5g}")))
else:
    display(Markdown("No Monte Carlo runs in this cell's datasheet."))

## Schematic vs post-layout
Two extractions of the same layout: **C** = kpex wire capacitance only, **RC** = Magic wire resistance + capacitance (the difference of the two
is the cost of the wiring resistance). Bars are the change of the typical value in % relative to the schematic; red = above the flag threshold.

In [ ]:
pl = {v: cd.postlayout(CELL, v) for v in ("c", "rc")}
pl = {k: v for k, v in pl.items() if v}
if pl:
    base = pl.get("c") or pl.get("rc")
    vars_ = [r["var"] for r in base["rows"] if r["delta_pct"] is not None]
    fig = go.Figure()
    for v, label, col in (("c", "post-layout C", "#2a6fdb"), ("rc", "post-layout R+C", "#d1495b")):
        if v not in pl:
            continue
        rows = {r["var"]: r for r in pl[v]["rows"]}
        ys = [rows[x]["delta_pct"] if x in rows else None for x in vars_]
        hov = [(f"{x}: schematic {rows[x]['schematic']['typ']:.5g} -> {rows[x]['postlayout']['typ']:.5g} {rows[x]['unit']} ({rows[x]['delta_pct']:+.1f} %)" if x in rows else "") for x in vars_]
        CLIP = 30.0   # a typical value near 0 (VOS) would otherwise flatten every other bar; the hover text has the true value
        ys = [None if y is None else max(-CLIP, min(CLIP, y)) for y in ys]
        fig.add_trace(go.Bar(y=vars_, x=ys, orientation="h", name=label, marker_color=col, hovertext=hov, hoverinfo="text+x"))
    thr = base["threshold_pct"]
    fig.update_layout(barmode="group", height=max(420, 24 * len(vars_) + 140), xaxis_title="typical value, post-layout vs schematic [%] (clipped at ±30 %, see hover)", xaxis_range=[-CLIP * 1.05, CLIP * 1.05],
                      title=dict(text="Post-layout change of the typical value", x=0.5), yaxis=dict(autorange="reversed"),
                      shapes=[dict(type="line", yref="paper", y0=0, y1=1, x0=x, x1=x, line=dict(color="#999", dash="dot")) for x in (-thr, thr)])
    fig.show()
    rows = []
    for r in base["rows"]:
        d = dict(var=r["var"], name=r["name"], unit=r["unit"], schematic=r["schematic"]["typ"])
        for v in pl:
            m = {q["var"]: q for q in pl[v]["rows"]}.get(r["var"])
            d[f"post {v}"] = m["postlayout"]["typ"] if m else None
            d[f"Δ% {v}"] = m["delta_pct"] if m else None
        rows.append(d)
    t = pd.DataFrame(rows)
    display(HTML(t.to_html(index=False, float_format=lambda x: f"{x:.5g}", na_rep="")))
else:
    display(Markdown("No post-layout comparison yet: `scripts/flow.sh %s`." % CELL))

## Provenance

In [ ]:
info = {"cell": CELL, "schematic run": ref, "flow": st and {k: st[k] for k in ("generated", "commit", "profile", "gds", "schematic")}}
for v in ("c", "rc"):
    r = cd.find_run(CELL, "pex", v)
    info[f"post-layout run ({v})"] = r
print(json.dumps(info, indent=1, default=str))